# Diagnose per-caption FP32 update differences
Select GPU T4 x2, enable Internet and attach the same four inputs. Save & Run All. Four short runs compare the old path against its own repeat, direct H5 with per-step logging, and direct H5 with buffered logging. Only eight full batches and tail27 are processed in each run. Reports include exact hashes, CUDA RNG and final tensor error magnitudes. Send the review ZIP.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil,tarfile
from pathlib import Path
REVISION='5eb012279e6fd0b5e47a56710ed5e3efa7449d2f'
MAPPING={'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/full_runtime.py': 'full_runtime.py', 'experiments/clip_patch_baseline/run_full.py': 'run_full.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/score_full.py': 'score_full.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/test_full_data.py': 'test_full_data.py', 'experiments/clip_patch_baseline/make_full_notebook.py': 'make_full_notebook.py', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/FULL_XE10_PROTOCOL.md': 'FULL_XE10_PROTOCOL.md', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/clip_patch_baseline/local_stage.py': 'local_stage.py', 'experiments/clip_patch_baseline/test_local_stage.py': 'test_local_stage.py', 'experiments/clip_patch_baseline/LOCAL_STAGE_V3_PROTOCOL.md': 'LOCAL_STAGE_V3_PROTOCOL.md', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/semantic_training.py': 'semantic_training.py', 'experiments/clip_patch_baseline/test_semantic_data.py': 'test_semantic_data.py', 'experiments/clip_patch_baseline/BANK_PROMPT_SCRATCH_PROTOCOL.md': 'BANK_PROMPT_SCRATCH_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json': 'SAVE_BANK_V130_PACKAGE.json', 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json': 'bank_words_corrected.json', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': 'FULL_PROMPT_FRESH_V175_RESULT.json', 'experiments/clip_patch_baseline/memory_fusion.py': 'memory_fusion.py', 'experiments/clip_patch_baseline/csa_full_data.py': 'csa_full_data.py', 'experiments/clip_patch_baseline/csa_full_runtime.py': 'csa_full_runtime.py', 'experiments/clip_patch_baseline/run_csa_full.py': 'run_csa_full.py', 'experiments/clip_patch_baseline/score_csa_full.py': 'score_csa_full.py', 'experiments/clip_patch_baseline/test_csa_full_data.py': 'test_csa_full_data.py', 'experiments/clip_patch_baseline/csa_patch_model.py': 'csa_patch_model.py', 'experiments/clip_patch_baseline/csa_sharing_data.py': 'csa_sharing_data.py', 'experiments/clip_patch_baseline/gradient_agreement.py': 'gradient_agreement.py', 'experiments/clip_patch_baseline/test_gradient_agreement.py': 'test_gradient_agreement.py', 'experiments/clip_patch_baseline/CSA_SHARED_FULL_XE10_PROTOCOL.md': 'CSA_SHARED_FULL_XE10_PROTOCOL.md', 'experiments/clip_patch_baseline/make_csa_full_notebook.py': 'make_csa_full_notebook.py'}
HASHES={'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/full_runtime.py': '230f7e10d96005fbffc33fb151a7a1d7c7245326e73826134a400772dadd04bb', 'experiments/clip_patch_baseline/run_full.py': '32dfb92228fad22d4f503680db1c8226db1e6abf9d7dd253786c3f16be9a97c0', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/score_full.py': 'fed170a6d12d1ab1bb969ae7eac29ee0855059417f6e2e876da193e279ca819a', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/test_full_data.py': 'f47160cf9cd85c96a4495dfa8264c0cd5e7cd60e8d5b6a77022932399e557431', 'experiments/clip_patch_baseline/make_full_notebook.py': '2417f0c691e291808b9f3b44f8bdb60090fcaf8bb487d71948538a7d61d5679d', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/FULL_XE10_PROTOCOL.md': 'a7c482bfe9efe3f23d4ebf235872204b8e0e0e9e69fae71e3a7b04b8980a986f', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/clip_patch_baseline/local_stage.py': '76ec1d4ace24fe2d95ea396d489692b7ff3c6289b71fd88c81b110cd2ad9426b', 'experiments/clip_patch_baseline/test_local_stage.py': '028e8bff388c44123368dac2aa21887d589adf1e02bb2d3f7082870564b9cfbb', 'experiments/clip_patch_baseline/LOCAL_STAGE_V3_PROTOCOL.md': '83ba0fd376a2f1f6069eca1a082c954e8316b236fa343807f813a40eb0a9487d', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'experiments/clip_patch_baseline/semantic_training.py': '001f05f3e6959570571135a55abb7bc740217a447371bab627af8c72e0f26d11', 'experiments/clip_patch_baseline/test_semantic_data.py': 'e9d9132db2e83b47e7f29cd1c465389cb4f16db2cd62a1cf27f048a06e29a572', 'experiments/clip_patch_baseline/BANK_PROMPT_SCRATCH_PROTOCOL.md': '93d1ebef1d359d393323951a152f50d16559c0fd235dac27eda4940653399a0c', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json': '826ba36628d6587860e2d548e950ea953bd4bbb8c0a8dc64393089dbc1e058e1', 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json': 'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': '452e94530b07b117be6873f004404b558e29e87cb09e97e27a1548d0bfd5f81a', 'experiments/clip_patch_baseline/memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'experiments/clip_patch_baseline/csa_full_data.py': '4028cd2b656d2244cabe2b4ab510b7a701bd8d462157adce62e10bf2060ca8c1', 'experiments/clip_patch_baseline/csa_full_runtime.py': '730a3282088480f56f8aaaee9af96a37a1bbb0c2f24256e19f14925f90c67486', 'experiments/clip_patch_baseline/run_csa_full.py': '86af1d1f9c2ed1352a7bc53b3f3d2fd8c00657b553dd34b870a2565e7efb072e', 'experiments/clip_patch_baseline/score_csa_full.py': '2a1ea2cf75dfe2ee400637d9094ccd504e06de0fb2dc0786e530bdee3e2528a5', 'experiments/clip_patch_baseline/test_csa_full_data.py': '6ab5adad7b3b5f5bbbb1648ef993e659c8309781ce5174bfb1971f6a4e72ec2c', 'experiments/clip_patch_baseline/csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'experiments/clip_patch_baseline/csa_sharing_data.py': '4d89428b8f833e34522bd51ec25640015edbebae4f0b5b81450058e2ebcb1243', 'experiments/clip_patch_baseline/gradient_agreement.py': 'dc06ffd922fccd9e14e860d07644dddcf320f7d9ac6410e3166cc4f53200971c', 'experiments/clip_patch_baseline/test_gradient_agreement.py': '2b2aa46091a5a752b3eab573f701c0f1e2849349d829e1c4d6a4d99ae73285a6', 'experiments/clip_patch_baseline/CSA_SHARED_FULL_XE10_PROTOCOL.md': '3b09530054a321f96691adb00cb4cb6340e42ce6eeba52b51268711c5df7ba0a', 'experiments/clip_patch_baseline/make_csa_full_notebook.py': '3cac3adbf862cbf7191aa36e0c89ae66c7337a4776eff73ce919d46f4455cfcc'}
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
BANK_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130')
PROMPT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
WALL_HOURS=1 # Parent stop cap, not ETA/billed quota. Do not auto-extend.
OUTPUT=Path('/kaggle/working/csa_per_caption_io_log_diagnostic_v2')
for p in (VISUAL_CACHE,COCO_JSON,BANK_ROOT,PROMPT_ROOT):
    if not p.exists():raise FileNotFoundError('Attach/correct inputs BEFORE GPU: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve previous output; use a fresh runtime/output path')
if shutil.disk_usage('/tmp').free<4*1024**3 or shutil.disk_usage('/kaggle/working').free<8*1024**3:raise RuntimeError('Need 4 GiB temporary scratch and 8 GiB Working')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='clip-patch-full10-',dir='/tmp'));DEPS=SOURCE/'deps';LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json'
cfg=dict(diagnostic_only=True,csa_per_caption_full=True,semantic=True,bank_root=str(BANK_ROOT),prompt_root=str(PROMPT_ROOT),prepare_seconds=1800,output=str(OUTPUT),stage=str(SOURCE/'stage'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),worker_seconds=600)
def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))
write_config()
def download(url,expected,path):
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent cap during download')
    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned asset: '+path.name)
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)

ACCEL_DEPS=SOURCE/'accelerate-deps'
os.environ['PYTHONPATH']=str(ACCEL_DEPS)+os.pathsep+str(SOURCE)

PILOT_SOURCES = {'probe_per_caption_io_log.py': '"""User-run, same-session parity and timing for per-caption FP32 XE."""\nimport hashlib\nimport sys\nimport time\nfrom datetime import timedelta\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom csa_full_data import order_for_epoch\nfrom full_data import CaptionBatches, FeatureReader, learning_rate, load_stage, loader, prepare\nfrom full_runtime import tensor_digest\nfrom real_bank_data import read, save\nfrom semantic_data import SemanticContext, stage_semantics\n\nTIMED_STEPS = 256\nAUDIT_STEPS = 8\n\n\nclass LocalProbeBatches(CaptionBatches):\n    def __getitem__(self, item):\n        if self.reader is None:\n            self.reader = FeatureReader(self.root)\n            root = Path(self.root)\n            self.reader.local = np.load(root/\'probe_patches.npy\', mmap_mode=\'r\', allow_pickle=False)\n            self.reader.local_rows = np.load(root/\'probe_rows.npy\', mmap_mode=\'r\', allow_pickle=False)\n        return super().__getitem__(item)\n\n\ndef digest(arrays):\n    h = hashlib.sha256()\n    for array in arrays:\n        array = np.ascontiguousarray(array)\n        h.update(str((array.shape, array.dtype)).encode())\n        h.update(array.tobytes())\n    return h.hexdigest()\n\n\ndef difference(actual, expected):\n    if actual.shape!=expected.shape:\n        raise ValueError(\'Tensor comparison shape mismatch\')\n    delta=actual.astype(np.float64)-expected.astype(np.float64)\n    norm=float(np.linalg.norm(expected.astype(np.float64)))\n    return dict(exact=bool(np.array_equal(actual,expected)),\n                max_abs=float(np.max(np.abs(delta))),\n                delta_l2=float(np.linalg.norm(delta)),reference_l2=norm,\n                relative_l2=float(np.linalg.norm(delta))/norm if norm else None)\n\n\ndef compare_records(reference, candidate):\n    fields=(\'input_sha256\',\'cuda_rng_sha256\',\'local_loss\',\'gradient_sha256\',\'weight_sha256\')\n    mismatches=[dict(step=i,fields=[key for key in fields if left[key]!=right[key]])\n                for i,(left,right) in enumerate(zip(reference[\'audit\'],candidate[\'audit\'],strict=True),1)]\n    return dict(initial_equal=reference[\'initial_sha256\']==candidate[\'initial_sha256\'],\n                mismatches=[row for row in mismatches if row[\'fields\']],\n                summed_nll_delta=candidate[\'summed_nll\']-reference[\'summed_nll\'],\n                final_weights_equal=reference[\'final_sha256\']==candidate[\'final_sha256\'])\n\n\ndef stage(cfg):\n    started = time.monotonic()\n    prepare(cfg)\n    stage_semantics(cfg, started+1500)\n    common_seconds = time.monotonic()-started\n    root = Path(cfg[\'stage\'])\n    reader = FeatureReader(root)\n    try:\n        order = order_for_epoch(reader.data, 1)\n        steps=AUDIT_STEPS if cfg.get(\'diagnostic_only\') else TIMED_STEPS\n        selected = np.unique(reader.data[\'caption_image\'][np.r_[order[:steps*32], order[-27:]]])\n        locations = reader.data[\'locations\'][selected]\n        selected = selected[np.lexsort((locations[:, 1], locations[:, 0]))]\n        row_map = np.full(len(reader.data[\'ids\']), -1, np.int64)\n        row_map[selected] = np.arange(len(selected))\n        patches = np.lib.format.open_memmap(root/\'probe_patches.npy\', mode=\'w+\', dtype=np.float16,\n                                            shape=(len(selected),196,768))\n        for ix, image in enumerate(selected):\n            value = reader.get(image)\n            patches[ix] = value\n            if not np.array_equal(patches[ix].view(np.uint16), value.view(np.uint16)):\n                raise ValueError(\'Local patch bytes differ\')\n        patches.flush()\n        del patches\n        np.save(root/\'probe_rows.npy\', row_map)\n    finally:\n        reader.close()\n    save(Path(cfg[\'output\'])/\'stage_timing.json\', dict(common_seconds=common_seconds,\n         subset_copy_seconds=time.monotonic()-started-common_seconds, subset_images=len(selected)))\n\n\ndef benchmark(cfg):\n    import torch\n    import torch.distributed as dist\n    from accelerate import Accelerator\n    from accelerate.utils import DistributedDataParallelKwargs, InitProcessGroupKwargs\n    from torch.nn import functional as F\n    from csa_full_runtime import make_model\n\n    torch.set_num_threads(1)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    accelerator = Accelerator(mixed_precision=\'no\', dynamo_backend=\'no\', kwargs_handlers=[\n        DistributedDataParallelKwargs(broadcast_buffers=False),\n        InitProcessGroupKwargs(backend=\'nccl\', timeout=timedelta(seconds=300))])\n    if accelerator.num_processes!=2 or accelerator.device.type!=\'cuda\':\n        raise ValueError(\'Select T4 x2\')\n    rank, device = accelerator.process_index, accelerator.device\n    root, out = Path(cfg[\'stage\']), Path(cfg[\'output\'])\n    data, words, _ = load_stage(root, verify=rank==0)\n    order = order_for_epoch(data, 1)\n    context = SemanticContext(root)\n\n    def run(mode, audit, snapshot=None):\n        torch.manual_seed(42)\n        raw = make_model(len(words), cfg).to(device)\n        model = accelerator.prepare_model(raw)\n        initial_sha=tensor_digest(raw) if audit else None\n        optimizer = accelerator.prepare_optimizer(torch.optim.Adam(raw.parameters(),lr=8e-6,betas=(.9,.99),eps=1e-8,weight_decay=0))\n        torch.manual_seed(4242+rank)\n        torch.cuda.manual_seed(4242+rank)\n        events = np.r_[order[:AUDIT_STEPS*32],order[-27:]] if audit else order[:TIMED_STEPS*32]\n        dataset = (LocalProbeBatches if mode==\'old\' else CaptionBatches)(root, events, rank)\n        pending, losses, audit_rows = [], [], []\n        torch.cuda.synchronize(device)\n        dist.barrier(device_ids=[rank])\n        started = time.monotonic()\n        for step,(x,seq,ev) in enumerate(loader(dataset),1):\n            input_sha = digest((x.numpy(), seq.numpy(), ev.numpy())) if audit else None\n            rng_sha=digest([torch.cuda.get_rng_state(device).cpu().numpy()]) if audit else None\n            x=x.to(device,non_blocking=True);seq=seq.to(device,non_blocking=True)\n            for group in optimizer.param_groups:group[\'lr\']=learning_rate(step)\n            optimizer.zero_grad(set_to_none=True)\n            logits=model(x,seq[:,:-1],context.events(data,ev,device))\n            count=torch.tensor([int((seq[:,1:]!=0).sum()),len(ev)],device=device)\n            dist.all_reduce(count)\n            nll=F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\')\n            loss=nll*(2/int(count[0]))\n            if not torch.isfinite(loss):raise ValueError(\'Nonfinite XE\')\n            accelerator.backward(loss)\n            norm=accelerator.clip_grad_norm_(raw.parameters(),1)\n            if not torch.isfinite(norm):raise ValueError(\'Nonfinite gradient norm\')\n            optimizer.step()\n            if mode in (\'old\',\'h5_step\'):\n                total=nll.detach().double()\n                dist.all_reduce(total)\n                losses.append(float(total))\n            else:\n                pending.append(nll.detach().double())\n                if step%100==0 or step==len(dataset):\n                    totals=torch.stack(pending)\n                    dist.all_reduce(totals)\n                    losses.extend(totals.cpu().tolist())\n                    pending.clear()\n            if audit:\n                audit_rows.append(dict(input_sha256=input_sha,cuda_rng_sha256=rng_sha, local_loss=float(nll.detach()),\n                    gradient_sha256=digest(p.grad.detach().cpu().numpy() for p in raw.parameters()),\n                    weight_sha256=tensor_digest(raw)))\n        torch.cuda.synchronize(device)\n        elapsed=torch.tensor(time.monotonic()-started,device=device,dtype=torch.float64)\n        dist.all_reduce(elapsed,op=dist.ReduceOp.MAX)\n        if len(losses)!=len(dataset) or pending:\n            raise ValueError(\'Incomplete loss accounting\')\n        record=dict(mode=mode,seconds=float(elapsed),events=len(events),updates=len(dataset),\n                    final_sha256=tensor_digest(raw),loss_sha256=digest([np.asarray(losses,np.float64)]),\n                    summed_nll=sum(losses),audit=audit_rows)\n        if audit:record[\'initial_sha256\']=initial_sha\n        if snapshot is not None and rank==0:\n            directory=root/\'diagnostic_tensors\'\n            directory.mkdir(exist_ok=True)\n            errors={}\n            for ix,(name,parameter) in enumerate(raw.named_parameters()):\n                for kind,tensor in ((\'weight\',parameter),(\'gradient\',parameter.grad)):\n                    array=tensor.detach().cpu().numpy()\n                    path=directory/f\'{ix}_{kind}.npy\'\n                    if snapshot==\'write\':\n                        np.save(path,array)\n                    else:\n                        expected=np.load(path,mmap_mode=\'r\',allow_pickle=False)\n                        stats=difference(array,expected)\n                        if not stats[\'exact\']:errors[name+\'/\'+kind]=stats\n                        del expected\n            record[\'tensor_errors\']=errors\n        del model,raw,optimizer\n        accelerator.free_memory()\n        return record\n\n    try:\n        if cfg.get(\'diagnostic_only\'):\n            records=[run(\'old\',True,\'write\'),run(\'old\',True,\'compare\'),\n                     run(\'h5_step\',True,\'compare\'),run(\'optimized\',True,\'compare\')]\n            comparisons=[compare_records(records[0],row) for row in records[1:]]\n            save(out/f\'diagnostic_rank{rank}.json\',dict(complete=True,records=records,comparisons=comparisons))\n            dist.barrier(device_ids=[rank])\n            if rank==0:\n                save(out/\'result.json\',dict(complete=True,diagnostic_only=True,\n                    ranks=[read(out/f\'diagnostic_rank{r}.json\') for r in (0,1)]))\n            return\n        old, optimized = run(\'old\',True),run(\'optimized\',True)\n        parity = all(old[k]==optimized[k] for k in (\'audit\',\'loss_sha256\',\'final_sha256\'))\n        save(out/f\'parity_rank{rank}.json\',dict(passed=parity,old=old,optimized=optimized))\n        flag=torch.tensor(int(parity),device=device)\n        dist.all_reduce(flag,op=dist.ReduceOp.MIN)\n        if not int(flag):raise ValueError(\'Input/loss/gradient/weight parity failed\')\n        runs=[run(mode,False) for mode in (\'old\',\'optimized\',\'optimized\',\'old\')]\n        if any((r[\'loss_sha256\'],r[\'final_sha256\'])!=(runs[0][\'loss_sha256\'],runs[0][\'final_sha256\']) for r in runs):\n            raise ValueError(\'Timed runs diverged\')\n        save(out/f\'timing_rank{rank}.json\',dict(passed=True,runs=runs))\n        dist.barrier(device_ids=[rank])\n        if rank==0:\n            save(out/\'result.json\',dict(complete=True,runs=runs,fp32=True,feature_sharing=False,\n                optimized_over_old_throughput=(runs[0][\'seconds\']+runs[3][\'seconds\'])/(runs[1][\'seconds\']+runs[2][\'seconds\'])))\n    finally:\n        accelerator.end_training()\n\n\nif __name__==\'__main__\':\n    {\'stage\':stage,\'bench\':benchmark}[sys.argv[1]](read(Path(sys.argv[2])))\n'}


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Parent cap exceeded')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Hybrid XE pilot active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Phase/parent cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    phase_times = {}
    setup_start = time.monotonic()
    sources = {name: HASHES[name] for name, destination in MAPPING.items() if destination not in PILOT_SOURCES}
    for name, digest in sources.items():
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+name, digest, SOURCE/MAPPING[name])
    for name, source in PILOT_SOURCES.items():
        (SOURCE/name).write_bytes(source.encode('utf-8'))
    embedded_sha = {name: hashlib.sha256((SOURCE/name).read_bytes()).hexdigest() for name in PILOT_SOURCES}
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION, sha256=sources, embedded_sha256=embedded_sha), indent=2))
    run([sys.executable,'-u',SOURCE/'test_full_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_local_stage.py'],120)
    run([sys.executable,'-u',SOURCE/'test_semantic_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_csa_full_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_gradient_agreement.py'],120)
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],60)

    run([sys.executable,'-m','pip','download','-q','--no-deps','--only-binary=:all:','--dest',SOURCE,'accelerate==1.15.0'],180)
    wheel=SOURCE/'accelerate-1.15.0-py3-none-any.whl'
    if hashlib.sha256(wheel.read_bytes()).hexdigest()!='97eacca0b73e45cb867dbf8c5d5d4dc32219544300e0c8992c7334dc2ef33cec':raise ValueError('Changed Accelerate wheel')
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',ACCEL_DEPS,wheel],180)
    run([sys.executable,'-c',"import accelerate;assert accelerate.__version__=='1.15.0';print('PinnedAccelerate',accelerate.__version__,flush=True)"],120)
    run([sys.executable,'-c',"from csa_patch_model import CSAPatchCaptioner;from csa_full_runtime import make_model;print('Neural module imports PASS; no model instantiated',flush=True)"],120)
    (OUTPUT/'accelerate_install.json').write_text(json.dumps(dict(version='1.15.0',wheel_sha256=hashlib.sha256(wheel.read_bytes()).hexdigest(),dependencies_reinstalled=False)))

    script=SOURCE/'probe_per_caption_io_log.py'
    run([sys.executable,'-u',script,'stage',CONFIG],1500)
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    run([*launch,script,'bench',CONFIG],1500)
    result=json.loads((OUTPUT/'result.json').read_text())
    if not result['complete'] or not result['diagnostic_only'] or len(result['ranks'])!=2:
        raise ValueError('Incomplete diagnostic')
    for rank,diagnostic in enumerate(result['ranks']):
        if len(diagnostic['records'])!=4 or len(diagnostic['comparisons'])!=3:
            raise ValueError('Missing repeat or isolation controls')
        for record in diagnostic['records']:
            if record['updates']!=9 or record['events']!=283 or len(record['audit'])!=9:
                raise ValueError('Incorrect diagnostic exposure')
        print('RANK',rank,'COMPARISONS',diagnostic['comparisons'],flush=True)
    complete=True
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc());raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,diagnostic_updates_per_run=9,precision="fp32",test_used=False,SCST=False,reranking=False)))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.rglob('*'):
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',review,flush=True);print('KEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    print('PER-CAPTION DIAGNOSTIC COMPLETE; inspect comparisons before adoption.' if complete else 'INCOMPLETE; preserve weights/logs; no automatic restart or partial-model ranking.',flush=True)
